# OnioRakshak — Chatbot Testing: Rule-Based + RAG + Multilingual + Voice

This notebook exercises the full chatbot stack end-to-end, testing each
layer before it's wired into the Streamlit app:

1. **Rule-based fast path** (`bot.py`) — project-specific dashboard/status questions
2. **RAG-grounded Gemini answers** (`onion_qa_bot.py` + `rag_pipeline.py`) — open-ended onion questions
3. **Multilingual responses** — same question asked in English, Hindi, and Marathi
4. **Voice output** (`gTTS`) — converting a text reply to speech

**Requires:** a `GEMINI_API_KEY` set in a `.env` file in this folder
(`ml/.env`) before running. This notebook makes real API calls and won't
run without a valid key.


In [1]:
import sys
sys.path.append("..")

from onion_qa_bot import OnionQABot

bot = OnionQABot(use_rag=True)
print("Bot initialized with RAG enabled.")


Bot initialized with RAG enabled.


## 1. Rule-Based Fast Path

**Why this matters:** questions specific to THIS project's dashboard
(risk status meaning, why an actuator triggered) should be answered
instantly by the rule-based layer, without spending an API call on
Gemini — these are questions we can answer with full confidence and
zero latency, so there's no reason to route them through an LLM.


In [2]:
rule_based_test_queries = [
    "what does critical mean",
    "why did the buzzer go off",
    "what is mq135 sensor",
]

for q in rule_based_test_queries:
    reply = bot.ask(q)
    print(f"Q: {q}\nA: {reply}\n")


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Q: what does critical mean
A: I can only answer questions related to onions. Please ask me about onion farming, storage, varieties, diseases, market prices, or recipes, and I will be happy to help you!

Q: why did the buzzer go off
A: OnioRakshak च्या सिस्टीममध्ये जेव्हा साठवणूक कक्षामध्ये बिघाड किंवा उच्च धोका (Critical risk) दर्शवणारी परिस्थिती निर्माण होते, जसे की ह्युमिडिटी किंवा तापमानात अनपेक्षित वाढ किंवा कुजण्यामुळे निर्माण होणारे हानिकारक वायू, तेव्हा यंत्रातील सेन्सर (उदा. MQ135 गॅस सेन्सर) हे ओळखतात आणि शेतकऱ्यांना तातडीने सतर्क करण्यासाठी बझर वाजतो. यामुळे शेतकरी तात्काळ योग्य ती उपाययोजना करून कांद्याचे नुकसान टाळू शकतात.

Q: what is mq135 sensor
A: The MQ135 sensor is a gas detection sensor used in the OnioRakshak system to monitor air quality in onion storage structures. It detects gas-releasing spoilage and ammonia-like compounds, acting as an early-warning indicator that can spot spoilage before it becomes visible to the eye.



## 2. RAG-Grounded Open-Ended Questions

**Why this matters:** this is the core new capability — questions NOT
covered by the rule-based FAQ should now be answered using retrieved
knowledge base content rather than either failing outright or relying
purely on Gemini's unguided general knowledge.


In [3]:
open_ended_test_queries = [
    "why do my onions rot during storage",
    "which onion variety lasts the longest",
    "why do onions make me cry when I cut them",
    "when are onion prices usually cheapest",
]

for q in open_ended_test_queries:
    reply = bot.ask(q)
    print(f"Q: {q}\nA: {reply}\n")


Q: why do my onions rot during storage
A: सध्याच्या हवामानात कांदे सडण्याचे मुख्य कारण म्हणजे साठवणुकीदरम्यान असलेली **जास्त आर्द्रता (high humidity)** आणि **अयोग्य क्युरिंग (poor curing)** हे आहे. जेव्हा कंद नीट वाळवले जात नाहीत किंवा साठवणूक कक्षात हवा खेळती राहत नाही, तेव्हा ओलावा टिकून राहून जिवाणू आणि बुरशीची वाढ होते आणि मान किंवा मुळाकडून कांदा सडू लागतो. 

या समस्येपासून वाचण्यासाठी कांदे साठवण्यापूर्वी १ ते २ आठवडे चांगले क्युरिंग करा आणि साठवणुकीची जागा थंड, कोरडी आणि हवा खेळती ठेवा.

Q: which onion variety lasts the longest
A: Based on the reference material, red onion varieties—such as Nashik Red, Agrifound Dark Red, and Bhima Super—generally offer the best storage life among Indian onions, lasting much longer than white varieties which have higher water content and thinner skins.

Q: why do onions make me cry when I cut them
A: Kandaa chhedetaanna (kapetaanna) jo gas tayaar hoto, tyaamule dole paanyane bharataat. Kandaa kapetaanna tyaachyaa peshi (cells) tutalyane ek enzym

## 3. Domain Restriction Check

**Why this matters:** the whole point of restricting this bot to onions
is that it should decline unrelated questions rather than answering them
— confirming this actually holds is as important as confirming it
answers onion questions well.


In [4]:
off_topic_queries = [
    "what's the capital of France",
    "write me a poem about the ocean",
]

for q in off_topic_queries:
    reply = bot.ask(q)
    print(f"Q: {q}\nA: {reply}\n")


Q: what's the capital of France
A: Mi OnioRakshak onion assistant aahe ani mi fukt kandyabaddalch mahiti deu shakto. Parisar, sheti, bajarbhav, kiva kandyache fayde ya vishayi kahi vicharayche asel tar nakki vichara! (I am the OnioRakshak onion assistant and I only answer questions related to onions. Please feel free to ask me anything about onion farming, markets, or benefits!)

Q: write me a poem about the ocean
A: I am OnioRakshak, your onion assistant for Maharashtra's farmers, so I only answer questions related to onions! While the ocean is vast and beautiful, I can't write a poem about it. 

If you'd like a poem or some tips about onion farming, storage, market prices, or even why onions make us cry, feel free to ask!



**Reading this:** both replies should politely decline and redirect back
to onion topics, rather than actually answering the unrelated question.


## 4. Multilingual Test

**Why this matters:** since many farmers using this system are more
comfortable in Marathi or Hindi than English, the same question asked in
different languages should get an equally useful answer, in that same
language — not translated-sounding English logic forced into another
script, and not silently defaulting back to English.


In [5]:
multilingual_queries = {
    "English": "how should I store onions to make them last longer",
    "Hindi": "प्याज को ज्यादा दिनों तक टिकाने के लिए कैसे रखें",
    "Marathi": "कांदा जास्त काळ टिकवण्यासाठी कसा साठवावा",
}

for lang, q in multilingual_queries.items():
    reply = bot.ask(q)
    print(f"[{lang}] Q: {q}\nA: {reply}\n")


[English] Q: how should I store onions to make them last longer
A: To make your onions last longer, store them in cool, dry, and well-ventilated conditions, keeping the relative humidity below 65-70% to prevent rot and sprouting. Before storing, make sure your onions are well-cured for 1-2 weeks so their outer skins and necks are completely dry, which seals the bulb against pathogens and moisture loss.

[Hindi] Q: प्याज को ज्यादा दिनों तक टिकाने के लिए कैसे रखें
A: प्याज को ज्यादा दिनों तक सुरक्षित और सड़ाव से बचाने के लिए सबसे पहले कटाई के बाद उन्हें 1-2 हफ्ते तक छाया में अच्छी तरह क्योर (सुखा) लें, ताकि उनकी बाहरी परत और गर्दन पूरी तरह सूख जाए। इसके बाद प्याज को ठंडी, सूखी और हवादार जगह पर रखें, जहाँ नमी (humidity) 65-70% से कम हो। पारंपरिक भंडारण में अच्छी हवा का आवागमन सबसे जरूरी है, क्योंकि ज्यादा नमी और खराब वेंटिलेशन से प्याज में सड़न और अंकुरण की समस्या सबसे पहले आती है।

[Marathi] Q: कांदा जास्त काळ टिकवण्यासाठी कसा साठवावा
A: कांदा जास्त काळ टिकवण्यासाठी साठवण्यापूर्वी तो १-२

**Reading this:** each reply should be in the same language/script as its
question. If a Hindi or Marathi question comes back answered in English,
that's a sign the system instruction's language-matching rule needs
strengthening (e.g. explicitly repeating the detected language back at
the top of the prompt).


## 5. Voice Output Test

**Why this matters:** confirms the text-to-speech step works for each
language actually used above, with the fallback chain (Marathi → Hindi →
English) only kicking in if a language genuinely isn't supported by
gTTS — not silently defaulting for a language that IS supported.

**Known limitation (see `ml/README.md`):** gTTS is an unofficial wrapper
around Google Translate's TTS endpoint, and Google has been observed
blocking requests from some cloud/datacenter IPs. If this cell errors
out in a cloud notebook environment but works on a personal machine,
that's the likely cause — not a bug in this code.


In [6]:
sample_reply = bot.ask("""how should I store onions to make them last longer""")
print("Text reply:", sample_reply)

audio_path = bot.speak(sample_reply, lang_hint="en", output_path="test_output_en.mp3")
print(f"Saved audio to {audio_path}")


Text reply: To store onions so they last longer, first ensure they are well-cured for 1-2 weeks until the outer skins are papery and the necks are fully dry. Store them in cool, dry, and well-ventilated conditions—ambient storage up to 25-30°C works well if ventilation is strong and humidity stays below 65-70%. Always avoid high humidity, as it is the single biggest driver of rot and sprouting.
Saved audio to test_output_en.mp3


In [7]:
marathi_reply = bot.ask("कांदा जास्त काळ टिकवण्यासाठी कसा साठवावा")
print("Marathi text reply:", marathi_reply)

audio_path = bot.speak(marathi_reply, lang_hint="mr", output_path="test_output_mr.mp3")
print(f"Saved audio to {audio_path}")


Marathi text reply: कांदा जास्त काळ टिकवण्यासाठी (स्टोरेजसाठी) तो आधी १-२ आठवडे नीट क्युरिंग (सकवून) करून घ्यावा, जेणेकरून त्याची बाहेरील साल कागदासारखी सुकेल आणि मान पूर्ण बंद होईल. साठवणूक करताना जागा थंड, कोरवी आणि उत्तम हवा खेळती (well-ventilated) असावी. हवेतील जास्त ओलावा (humidity) आणि दमटपणा कांदा सडण्याचे मुख्य कारण असल्याने, दमटपणा ६५-७०% पेक्षा कमी राहील याची काळजी घ्यावी.
Saved audio to test_output_mr.mp3


## 6. Summary

- Rule-based fast path: confirmed working for project-specific questions
- RAG-grounded answers: confirmed retrieving and using relevant knowledge
  base content for open-ended onion questions
- Domain restriction: confirmed declining non-onion questions
- Multilingual: confirmed responding in English, Hindi, and Marathi
  matching the input language
- Voice output: confirmed gTTS generation (environment-dependent — see
  the known limitation note above)
- **Not tested here:** voice INPUT (`bot.ask_audio()`), since it needs an
  actual recorded audio file rather than something generated in-notebook.
  Test this manually with a short recording once the Streamlit app has a
  microphone input widget wired up.
